In [ ]:
using DifferentialEquations, OpenQuantumTools, Plots, LaTeXStrings, LinearAlgebra

$$H(s)/\hbar = (1-s)\sigma_x/2 + s\sigma_z/2 + \sigma_z*B+H_B $$
Bosonic Ohmic bath so spectral density (see *Quantum adiabatic markovian master equations*) $$ \gamma(\omega)=\eta\frac{2\pi\omega\exp(-|\omega|/\omega_c)}{1-\exp(-\beta\omega)}$$


In [ ]:
# Hamiltonian
H = DenseHamiltonian([(s)->1-s, (s)->s], -[σx, σz]/2, unit=:ħ)
# initial state
u0 = PauliVec[1][1]
# coupling
coupling = ConstantCouplings(["Z"], unit=:ħ)
# bath
bath = Ohmic(1e-4, 4, 16)
annealing = Annealing(H, u0; coupling=coupling, bath=bath)

tf = 60
U = solve_unitary(annealing, tf, alg=Tsit5(), abstol=1e-8, reltol=1e-8)
U = InplaceUnitary(U)

@time solr = solve_redfield(annealing, tf, U, alg=Tsit5())
# we set the integration error tolerance to 1e-5 for speed
@time solc = solve_cgme(annealing, tf, U, alg=Tsit5(), int_atol=1e-5, int_rtol=1e-5)
plot(solr, H, [0], 0:0.01:tf, linewidth=2, xlabel="t (ns)", ylabel="\$P_G(t)\$", label="Redfield")
plot!(solc, H, [0], 0:0.01:tf, linewidth=2, label="CGME")

In [ ]:
using QuadGK

In [ ]:

g(t) = quadgk((w)->sqrt(γ(w, bath))*exp(1.0im*w*t)/2/π, -Inf, Inf)[1]

t = range(-0.5,0.5,length=500)
g_value = g.(t)
c_value = [correlation(x, bath) for x in t];

plot(t, real.(g_value), label="Re[g(t)]", linewidth=2)
plot!(t, imag.(g_value), label="Im[g(t)]", linewidth=2)
plot!(t, real.(c_value), label="Re[C(t)]", linewidth=2)
plot!(t, imag.(c_value), label="Im[C(t)]", linewidth=2)
xlabel!("t (ns)")
ylabel!("correlation")

In [ ]:


t = range(-4,4,length=2000)
g_value = g.(t)
gf = construct_interpolations(t, g_value, extrapolation = "flat")

t = range(-0.5,0.5,length=500)
g_value = gf.(t)
plot(t, real.(g_value), label="Re[g(t)]", linewidth=2)
plot!(t, imag.(g_value), label="Im[g(t)]", linewidth=2)
xlabel!("t (ns)")
ylabel!("fitted jump correlator")


In [ ]:
ubath = ULEBath(gf)
annealing = Annealing(H, u0; coupling=coupling, bath=ubath)
@time solu = solve_ule(annealing, tf, U, alg=Tsit5(), int_atol=1e-5, int_rtol=1e-5)
plot(solr, H, [0], 0:0.01:tf, linewidth=2, xlabel="t (ns)", ylabel="\$P_G(t)\$", label="Redfield")
plot!(solc, H, [0], 0:0.01:tf, linewidth=2, label="CGME")
plot!(solu, H, [0], 0:0.01:tf, linewidth=2, label="ULE")

## We can try to simulate a damped oscillator

In [ ]:
basis_lim = 5
n = complex(diagm(0=>[i-1.0 for i in 1:basis_lim]))
a = complex(diagm(1=>[sqrt(i) for i in 1:basis_lim-1]))
at = complex(diagm(-1=>[sqrt(i) for i in 1:basis_lim-1]))
H_osc = DenseHamiltonian([(s)->1.0],[n], unit=:ħ)
u0_osc=complex([0.0,0.0,0.0,1.0,0.0])
coupling_osc = ConstantCouplings([11*a], unit=:ħ)

bath = Ohmic(1e-4, 4, 16)
annealing_osc = Annealing(H_osc, u0_osc, coupling=coupling_osc, bath=bath)

tf = 2
U_osc = solve_unitary(annealing_osc, tf, alg=Tsit5(), abstol=1e-8, reltol=1e-8)
U_osc = InplaceUnitary(U_osc);

In [ ]:
@time solr_osc = solve_redfield(annealing_osc, tf, U_osc, alg=Tsit5())
@time solc_osc = solve_cgme(annealing_osc, tf, U_osc, alg=Tsit5(), int_atol=1e-5, int_rtol=1e-5)

t_list_r=solr_osc.t
ρt_r=solr_osc.u

t_list_c=solc_osc.t
ρt_c=solc_osc.u;

In [ ]:
n_exp_r = real([tr(n*ρt_r[i]) for i in 1:length(t_list_r)])
n_exp_c = real([tr(n*ρt_c[i]) for i in 1:length(t_list_c)]);

In [ ]:
plot(t_list_r[1:end-60], n_exp_r[1:end-60], label="Redfield", linewidth=2, xlabel="t (ns)", ylabel="\$\\langle n \\rangle\$")
plot(t_list_c, n_exp_c, label="CGME", linewidth=2)

In [ ]:
annealing_osc_u = Annealing(H_osc, u0_osc; coupling=coupling_osc, bath=ubath)
@time solu_osc = solve_ule(annealing_osc_u, tf, U_osc, alg=Tsit5(), int_atol=1e-5, int_rtol=1e-5)

t_list_u=solu_osc.t
ρt_u=solu_osc.u;

In [ ]:
n_exp_u = real([tr(n*ρt_u[i]) for i in 1:length(t_list_u)]);

In [ ]:
n_0=3
th_test = n_0*exp.(-t_list_u*11/100);

In [ ]:
plot(t_list_r, n_exp_r, label="Redfield", linewidth=2, xlabel="t (ns)", ylabel="\$\\langle n \\rangle\$")
plot(t_list_c, n_exp_c, label="CGME", linewidth=2)
plot!(t_list_u, n_exp_u, label="ULE", linewidth=2)
plot!(t_list_u, th_test, label="Lindblad", linewidth=2)